# 1. Fill AAAI ACL CVPR EMNLP ICLR ICML IJCAI KDD SIGIR WWW
### 1-1. 先利用DOI（若有DOI），Title（没有DOI）去匹配id_openalex_papers_with_abstract_main_conference_updated.jsonl，匹配到的就保存到aaai_matched_openalex_with_abstract.jsonl、aaai_matched_openalex_without_abstract.jsonl文件中，没匹配到的保存到aaai_unmatched_dblp_papers.csv,下一步从openalex继续获取这些paper

In [2]:
import json
import re
import pandas as pd
from pathlib import Path


# =========================
# Config
# =========================
conference_name = "IJCAI"

BASE_DIR = Path("/home/user/GSK/lily/science_of_science/NewDataset")

DBLP_DIR = BASE_DIR / "AI_impact_on_cs_publications/Data/20260706_dblp"
DBLP_CSV = DBLP_DIR / f"{conference_name.lower()}_dblp_2020_2025_all_tracks.csv"

OPENALEX_JSONL = BASE_DIR / "Data/id_openalex_papers_with_abstract_main_conference_updated.jsonl"

OUT_DIR = BASE_DIR / "AI_impact_on_cs_publications/Data/20260708_openalex"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MATCHED_WITH_ABSTRACT_JSONL = OUT_DIR / f"{conference_name.lower()}_matched_openalex_with_abstract.jsonl"
MATCHED_WITHOUT_ABSTRACT_JSONL = OUT_DIR / f"{conference_name.lower()}_matched_openalex_without_abstract.jsonl"
MATCHED_MAP_CSV = OUT_DIR / f"{conference_name.lower()}_dblp_openalex_match_map.csv"
UNMATCHED_CSV = OUT_DIR / f"{conference_name.lower()}_unmatched_dblp_papers.csv"


# =========================
# Normalization helpers
# =========================
def normalize_doi(doi):
    if doi is None or pd.isna(doi):
        return ""

    doi = str(doi).strip().lower()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "")
    doi = doi.replace("http://dx.doi.org/", "")
    doi = doi.replace("doi:", "")
    doi = doi.strip().rstrip(".")
    return doi


def normalize_title(title):
    if title is None or pd.isna(title):
        return ""

    title = str(title).lower()
    title = re.sub(r"<[^>]+>", " ", title)
    title = re.sub(r"&amp;", " and ", title)
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_conference(conf):
    if conf is None or pd.isna(conf):
        return ""

    conf = str(conf).strip().upper()

    aliases = {
        "THEWEB": "WWW",
        "WWW": "WWW",
        "THE WEB CONF": "WWW",
        "WEB CONF": "WWW",
        "AAAI": "AAAI",
        "ACL": "ACL",
        "CVPR": "CVPR",
        "EMNLP": "EMNLP",
        "ICLR": "ICLR",
        "ICML": "ICML",
        "IJCAI": "IJCAI",
        "KDD": "KDD",
        "SIGIR": "SIGIR",
    }

    return aliases.get(conf, conf)


def parse_conference_year_from_dblp_source(dblp_source):
    if not dblp_source:
        return "", None

    if isinstance(dblp_source, dict):
        conf = (
            dblp_source.get("conference")
            or dblp_source.get("Conference")
            or dblp_source.get("venue")
            or dblp_source.get("conf")
            or ""
        )
        year = (
            dblp_source.get("year")
            or dblp_source.get("Year")
            or dblp_source.get("publication_year")
            or None
        )

        try:
            year = int(year) if year is not None and str(year).strip() else None
        except Exception:
            year = None

        return normalize_conference(conf), year

    text = str(dblp_source)

    conf = ""
    year = None

    match_conf = re.search(r"/conf/([^/]+)/", text)
    if not match_conf:
        match_conf = re.search(r"conf/([^/]+)/", text)

    if match_conf:
        conf = normalize_conference(match_conf.group(1))

    match_year = re.search(r"(20\d{2})", text)
    if match_year:
        year = int(match_year.group(1))

    return conf, year


def get_openalex_conference_year(item):
    conf, year = parse_conference_year_from_dblp_source(item.get("dblp_source"))

    if year is None:
        year = item.get("publication_year")
        try:
            year = int(year) if year is not None and str(year).strip() else None
        except Exception:
            year = None

    return conf, year


def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return ""

    word_positions = []
    for word, positions in inverted_index.items():
        for pos in positions:
            word_positions.append((pos, word))

    word_positions.sort(key=lambda x: x[0])
    return " ".join(word for _, word in word_positions)


def has_abstract(item):
    if item.get("reconstructed_abstract"):
        return True

    if item.get("abstract"):
        return True

    if item.get("abstract_inverted_index"):
        abstract_text = reconstruct_abstract(item.get("abstract_inverted_index"))
        if abstract_text:
            item["reconstructed_abstract"] = abstract_text
            return True

    return False


# =========================
# Load DBLP papers
# =========================
dblp_df = pd.read_csv(DBLP_CSV)

dblp_df["conference_norm"] = dblp_df["conference"].map(normalize_conference)
dblp_df["year_norm"] = pd.to_numeric(dblp_df["year"], errors="coerce").astype("Int64")
dblp_df["doi_norm"] = dblp_df["doi"].map(normalize_doi)
dblp_df["title_norm"] = dblp_df["title"].map(normalize_title)

dblp_df = dblp_df[dblp_df["conference_norm"] == normalize_conference(conference_name)].copy()

print(f"Loaded DBLP papers for {conference_name}: {len(dblp_df):,}")


# =========================
# Build OpenAlex indexes
# =========================
doi_index = {}
title_index = {}

n_openalex = 0

with OPENALEX_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        item = json.loads(line)
        n_openalex += 1

        doi_norm = normalize_doi(item.get("doi"))
        if doi_norm and doi_norm not in doi_index:
            doi_index[doi_norm] = item

        oa_conf, oa_year = get_openalex_conference_year(item)
        title_norm = normalize_title(item.get("title"))

        if oa_conf and oa_year and title_norm:
            key = (normalize_conference(oa_conf), int(oa_year), title_norm)
            if key not in title_index:
                title_index[key] = item

print(f"Loaded OpenAlex records: {n_openalex:,}")
print(f"DOI index size: {len(doi_index):,}")
print(f"Title index size: {len(title_index):,}")


# =========================
# Match DBLP papers to OpenAlex
# =========================
matched_with_abstract = []
matched_without_abstract = []
match_rows = []
unmatched_rows = []

seen_with_abstract_ids = set()
seen_without_abstract_ids = set()

for row_idx, row in dblp_df.iterrows():
    doi_norm = row["doi_norm"]
    title_norm = row["title_norm"]
    conf_norm = row["conference_norm"]
    year_norm = row["year_norm"]

    matched_item = None
    match_method = None

    if doi_norm:
        matched_item = doi_index.get(doi_norm)
        if matched_item is not None:
            match_method = "doi"

    if matched_item is None and title_norm and pd.notna(year_norm):
        title_key = (conf_norm, int(year_norm), title_norm)
        matched_item = title_index.get(title_key)
        if matched_item is not None:
            match_method = "conference_year_title"

    if matched_item is not None:
        openalex_id = matched_item.get("id", "")
        abstract_available = has_abstract(matched_item)

        match_rows.append({
            "dblp_row_index": row_idx,
            "conference": row.get("conference"),
            "year": row.get("year"),
            "title": row.get("title"),
            "doi": row.get("doi"),
            "openalex_id": openalex_id,
            "openalex_doi": matched_item.get("doi"),
            "openalex_title": matched_item.get("title"),
            "match_method": match_method,
            "has_abstract": abstract_available,
        })

        if abstract_available:
            if openalex_id and openalex_id not in seen_with_abstract_ids:
                matched_with_abstract.append(matched_item)
                seen_with_abstract_ids.add(openalex_id)
        else:
            if openalex_id and openalex_id not in seen_without_abstract_ids:
                matched_without_abstract.append(matched_item)
                seen_without_abstract_ids.add(openalex_id)

    else:
        out = row.to_dict()
        out["match_method"] = "unmatched"
        unmatched_rows.append(out)


matched_map_df = pd.DataFrame(match_rows)
unmatched_df = pd.DataFrame(unmatched_rows)

print(f"Matched DBLP rows: {len(matched_map_df):,}")
print(f"Matched with abstract records: {len(matched_with_abstract):,}")
print(f"Matched without abstract records: {len(matched_without_abstract):,}")
print(f"Unmatched DBLP rows: {len(unmatched_df):,}")

if len(dblp_df) > 0:
    print(f"DBLP-row match rate: {len(matched_map_df) / len(dblp_df) * 100:.2f}%")

if len(matched_map_df) > 0:
    print(
        "Abstract coverage among matched DBLP rows: "
        f"{matched_map_df['has_abstract'].mean() * 100:.2f}%"
    )


# =========================
# Save outputs
# =========================
with MATCHED_WITH_ABSTRACT_JSONL.open("w", encoding="utf-8") as f:
    for item in matched_with_abstract:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with MATCHED_WITHOUT_ABSTRACT_JSONL.open("w", encoding="utf-8") as f:
    for item in matched_without_abstract:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

matched_map_df.to_csv(MATCHED_MAP_CSV, index=False, encoding="utf-8-sig")
unmatched_df.to_csv(UNMATCHED_CSV, index=False, encoding="utf-8-sig")

print(f"Saved matched OpenAlex records with abstract to: {MATCHED_WITH_ABSTRACT_JSONL}")
print(f"Saved matched OpenAlex records without abstract to: {MATCHED_WITHOUT_ABSTRACT_JSONL}")
print(f"Saved match map to: {MATCHED_MAP_CSV}")
print(f"Saved unmatched DBLP papers to: {UNMATCHED_CSV}")


# =========================
# Optional diagnostics
# =========================
if not matched_map_df.empty:
    display(
        matched_map_df
        .groupby(["year", "match_method", "has_abstract"])
        .size()
        .reset_index(name="n")
        .sort_values(["year", "match_method", "has_abstract"])
    )

if not unmatched_df.empty:
    display(
        unmatched_df
        .groupby(["year", "track_type"])
        .size()
        .reset_index(name="n_unmatched")
        .sort_values(["year", "track_type"])
    )

Loaded DBLP papers for IJCAI: 5,698
Loaded OpenAlex records: 84,905
DOI index size: 83,579
Title index size: 79,484
Matched DBLP rows: 5,524
Matched with abstract records: 5,524
Matched without abstract records: 0
Unmatched DBLP rows: 174
DBLP-row match rate: 96.95%
Abstract coverage among matched DBLP rows: 100.00%
Saved matched OpenAlex records with abstract to: /home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Data/20260708_openalex/ijcai_matched_openalex_with_abstract.jsonl
Saved matched OpenAlex records without abstract to: /home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Data/20260708_openalex/ijcai_matched_openalex_without_abstract.jsonl
Saved match map to: /home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Data/20260708_openalex/ijcai_dblp_openalex_match_map.csv
Saved unmatched DBLP papers to: /home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Data/20260708_openale

,year,match_method,has_abstract,n
0,2020,doi,True,777
1,2021,doi,True,720
2,2022,conference_year_title,True,1
3,2022,doi,True,862
4,2023,doi,True,846
5,2024,conference_year_title,True,81
6,2024,doi,True,957
7,2025,doi,True,1280


,year,track_type,n_unmatched
0,2020,main_track,1
1,2020,non-main_track,42
2,2021,non-main_track,40
3,2022,non-main_track,66
4,2023,non-main_track,7
5,2024,non-main_track,18


## 1-2.从OpenAlex获取没匹配到的paper

In [ ]:
import json
import os
import re
import time
import random
from difflib import SequenceMatcher
import pandas as pd
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    from IPython.display import display
except ImportError:
    display = print


# =========================
# Config
# =========================
conference_name = "ICLR"

BASE_DIR = Path("/home/user/GSK/lily/science_of_science/NewDataset")

OUT_DIR = BASE_DIR / "AI_impact_on_cs_publications/Train_Data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

UNMATCHED_CSV = OUT_DIR / f"{conference_name.lower()}_dblp_2019_all_tracks.csv"

OPENALEX_API_KEY = ""  # Fill via environment variable.
OPENALEX_MAILTO = ""
OPENALEX_WORKS_URL = "https://api.openalex.org/works"
OPENALEX_AUTOCOMPLETE_WORKS_URL = "https://api.openalex.org/autocomplete/works"
MAX_YEAR_DIFFERENCE = 2
MIN_TITLE_SIMILARITY = 0.96
AUTOCOMPLETE_MAX_CANDIDATES = 10
SEARCH_MAX_CANDIDATES = 25
TITLE_STRATEGY_VERSION = "autocomplete_without_per_page_v2"

DOI_BATCH_SIZE = 50
TITLE_MAX_WORKERS = 1
SLEEP_SECONDS = 1.2

# Cache policy:
# found: request succeeded and item was matched, do not request again.
# not_found: request succeeded but no verified item matched, do not request again by default.
# failed: request did not complete successfully, retry by default.
RETRY_NOT_FOUND = False
RETRY_FAILED = True

REFETCH_CACHE_JSON = OUT_DIR / (
    f"{conference_name.lower()}_openalex_cache.json"
)

REFETCHED_WITH_ABSTRACT_JSONL = OUT_DIR / f"{conference_name.lower()}_openalex_with_abstract.jsonl"
REFETCHED_WITHOUT_ABSTRACT_JSONL = OUT_DIR / f"{conference_name.lower()}_openalex_without_abstract.jsonl"
REFETCH_MATCH_MAP_CSV = OUT_DIR / f"{conference_name.lower()}_match_map.csv"
STILL_UNMATCHED_CSV = OUT_DIR / f"{conference_name.lower()}_unmatched_after_openalex.csv"


# =========================
# Basic helpers
# =========================
def normalize_doi(doi):
    if doi is None or pd.isna(doi):
        return ""

    doi = str(doi).strip().lower()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "")
    doi = doi.replace("http://dx.doi.org/", "")
    doi = doi.replace("doi:", "")
    doi = doi.strip().rstrip(".")
    return doi


def format_openalex_doi(doi):
    doi_norm = normalize_doi(doi)
    if not doi_norm:
        return ""
    return f"https://doi.org/{doi_norm}"


def normalize_title(title):
    if title is None or pd.isna(title):
        return ""

    title = str(title).lower()
    title = re.sub(r"<[^>]+>", " ", title)
    title = re.sub(r"&amp;", " and ", title)
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def clean_title_for_openalex_search(title, max_len=250):
    if title is None or pd.isna(title):
        return ""

    title = str(title)
    title = re.sub(r"<[^>]+>", " ", title)
    title = title.replace("\n", " ").replace("\r", " ")

    # OpenAlex treats ? and * as wildcard characters in search.title.
    # Remove them to avoid 400 Bad Request in stemmed search.
    title = title.replace("?", " ")
    title = title.replace("*", " ")

    title = re.sub(r"\s+", " ", title).strip()
    title = title.strip(" .,:;")

    if len(title) > max_len:
        title = title[:max_len].strip()

    return title


def safe_year_value(year):
    if year is None or pd.isna(year):
        return None

    try:
        year = int(float(year))
    except Exception:
        return None

    if 1900 <= year <= 2100:
        return year

    return None


def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return ""

    word_positions = []

    for word, positions in inverted_index.items():
        for pos in positions:
            word_positions.append((pos, word))

    word_positions.sort(key=lambda x: x[0])
    return " ".join(word for _, word in word_positions)


def has_abstract(item):
    if not item:
        return False

    if item.get("reconstructed_abstract"):
        return True

    if item.get("abstract"):
        return True

    if item.get("abstract_inverted_index"):
        abstract_text = reconstruct_abstract(item.get("abstract_inverted_index"))
        if abstract_text:
            item["reconstructed_abstract"] = abstract_text
            return True

    return False


def chunk_list(values, chunk_size):
    values = list(values)
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


# =========================
# Cache helpers
# =========================
def load_cache(cache_path):
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as f:
            return json.load(f)

    return {}


def save_cache(cache, cache_path):
    tmp_path = cache_path.with_suffix(".tmp")

    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)

    tmp_path.replace(cache_path)


def should_fetch_cache_entry(entry):
    """
    Decide whether a cached query should be fetched again.

    status:
    - found: request succeeded and item was found.
    - not_found: request succeeded but no item matched.
    - failed: request did not complete successfully.
    """
    if entry is None:
        return True

    status = entry.get("status")

    if status == "found":
        return False

    if status == "not_found":
        return RETRY_NOT_FOUND

    if status == "failed":
        return RETRY_FAILED

    # Compatibility with older cache files that only used found=True/False.
    if entry.get("found") is True:
        return False

    if entry.get("found") is False:
        # Old cache cannot distinguish not_found from request failure.
        # Retry once under the new status-aware logic.
        return True

    return True


# =========================
# OpenAlex request helpers
# =========================
class OpenAlexBadRequest(RuntimeError):
    pass


def openalex_get(params, url=OPENALEX_WORKS_URL, timeout=60, max_retry=8):
    params = dict(params)

    if OPENALEX_API_KEY:
        params["api_key"] = OPENALEX_API_KEY
    if OPENALEX_MAILTO:
        params["mailto"] = OPENALEX_MAILTO

    last_error = None

    for attempt in range(1, max_retry + 1):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=timeout
            )

            if response.status_code == 400:
                print("[400 Bad Request]")
                print("URL:", response.url)
                print("Response:", response.text[:1000])
                raise OpenAlexBadRequest(f"400 Bad Request: {response.text[:300]}")

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    wait = int(retry_after) + random.uniform(1, 3)
                else:
                    wait = min(300, 30 * attempt + random.uniform(5, 10))

                print(f"[429] Rate limited. Wait {wait:.1f}s")
                time.sleep(wait)
                last_error = RuntimeError("429 Too Many Requests")
                continue

            response.raise_for_status()
            return response.json()

        except OpenAlexBadRequest:
            raise

        except Exception as exc:
            last_error = exc

            if attempt < max_retry:
                wait = min(120, 3 * attempt + random.uniform(1, 3))
                print(f"[Retry {attempt}/{max_retry}] {exc}; wait {wait:.1f}s")
                time.sleep(wait)

    raise RuntimeError(f"OpenAlex request failed after {max_retry} retries: {last_error}")


def fetch_openalex_by_single_doi(doi):
    doi_value = format_openalex_doi(doi)

    if not doi_value:
        return None

    data = openalex_get({
        "filter": f"doi:{doi_value}",
        "per_page": 1,
    })

    results = data.get("results", [])
    return results[0] if results else None


def openalex_work_id(value):
    """Convert an OpenAlex URL or short ID to W... form."""
    text = str(value or "").strip().rstrip("/")
    return text.rsplit("/", 1)[-1]


def fetch_openalex_work_by_id(value):
    work_id = openalex_work_id(value)
    if not work_id:
        return None
    return openalex_get({}, url=f"{OPENALEX_WORKS_URL}/{work_id}")


def verify_title_candidates(candidates, query_title, query_year):
    """Return the best candidate after strict title/year validation."""
    query_norm = normalize_title(query_title)
    verified = []

    for item in candidates:
        candidate_norm = normalize_title(item.get("title") or item.get("display_name"))
        if not candidate_norm:
            continue

        similarity = SequenceMatcher(None, query_norm, candidate_norm).ratio()
        exact = candidate_norm == query_norm
        if not exact and similarity < MIN_TITLE_SIMILARITY:
            continue

        candidate_year = safe_year_value(item.get("publication_year"))
        if query_year is not None:
            # Strong year check: candidates with no year are rejected.
            if candidate_year is None:
                continue
            year_difference = abs(candidate_year - query_year)
            if year_difference > MAX_YEAR_DIFFERENCE:
                continue
        else:
            year_difference = None

        verified.append({
            "item": item,
            "exact": exact,
            "similarity": similarity,
            "year_difference": year_difference,
        })

    if not verified:
        return None

    # Prefer exact titles, then the closest year, then the highest title score.
    verified.sort(
        key=lambda value: (
            0 if value["exact"] else 1,
            value["year_difference"] if value["year_difference"] is not None else 0,
            -value["similarity"],
        )
    )
    return verified[0]


def autocomplete_title_candidates(query_title):
    """Autocomplete returns compact objects, so hydrate each candidate by ID."""
    data = openalex_get(
        {
            "q": clean_title_for_openalex_search(query_title),
        },
        url=OPENALEX_AUTOCOMPLETE_WORKS_URL,
    )

    hydrated = []
    seen_ids = set()
    hydration_errors = []

    # The autocomplete endpoint has a fixed result size and does not accept
    # the Works API's `per_page` parameter. Limit candidates locally instead.
    for candidate in data.get("results", [])[:AUTOCOMPLETE_MAX_CANDIDATES]:
        candidate_id = candidate.get("id")
        work_id = openalex_work_id(candidate_id)
        if not work_id or work_id in seen_ids:
            continue
        seen_ids.add(work_id)

        try:
            work = fetch_openalex_work_by_id(work_id)
            if work:
                hydrated.append(work)
        except Exception as exc:
            hydration_errors.append(f"{work_id}: {exc}")

    return hydrated, hydration_errors


def search_title_candidates(query_title, query_year):
    params = {
        "search": clean_title_for_openalex_search(query_title),
        "per_page": SEARCH_MAX_CANDIDATES,
    }
    if query_year is not None:
        lower = query_year - MAX_YEAR_DIFFERENCE
        upper = query_year + MAX_YEAR_DIFFERENCE
        params["filter"] = f"publication_year:{lower}-{upper}"

    data = openalex_get(params)
    return data.get("results", [])


def get_best_title_match(query_title, year=None):
    query_norm = normalize_title(query_title)
    search_title = clean_title_for_openalex_search(query_title)
    query_year = safe_year_value(year)

    base_result = {
        "status": "not_found",
        "item": None,
        "match_method": "empty_title",
        "error": None,
        "title_similarity": None,
        "year_difference": None,
        "fallback_reason": None,
    }
    if not query_norm or not search_title:
        return base_result

    fallback_reasons = []

    # 1) Prefer the same autocomplete endpoint used by the OpenAlex search box.
    try:
        autocomplete_candidates, hydration_errors = autocomplete_title_candidates(query_title)
        if hydration_errors:
            fallback_reasons.append(
                "autocomplete_hydration_errors=" + " | ".join(hydration_errors)
            )
        best = verify_title_candidates(
            autocomplete_candidates,
            query_title=query_title,
            query_year=query_year,
        )
        if best:
            return {
                "status": "found",
                "item": best["item"],
                "match_method": (
                    "autocomplete_title_exact"
                    if best["exact"]
                    else "autocomplete_title_fuzzy"
                ),
                "error": None,
                "title_similarity": best["similarity"],
                "year_difference": best["year_difference"],
                "fallback_reason": None,
            }
        fallback_reasons.append("autocomplete_no_verified_match")
    except Exception as exc:
        fallback_reasons.append(f"autocomplete_request_failed: {exc}")

    # 2) Fall back to Works search when autocomplete fails or has no valid match.
    try:
        search_candidates = search_title_candidates(query_title, query_year)
        best = verify_title_candidates(
            search_candidates,
            query_title=query_title,
            query_year=query_year,
        )
        if best:
            return {
                "status": "found",
                "item": best["item"],
                "match_method": (
                    "search_title_exact" if best["exact"] else "search_title_fuzzy"
                ),
                "error": None,
                "title_similarity": best["similarity"],
                "year_difference": best["year_difference"],
                "fallback_reason": "; ".join(fallback_reasons) or None,
            }
        fallback_reasons.append("search_no_verified_match")
        return {
            **base_result,
            "match_method": "autocomplete_then_search_not_found",
            "error": "; ".join(fallback_reasons),
            "fallback_reason": "; ".join(fallback_reasons),
        }
    except Exception as exc:
        fallback_reasons.append(f"search_request_failed: {exc}")
        return {
            **base_result,
            "status": "failed",
            "match_method": "autocomplete_then_search_failed",
            "error": "; ".join(fallback_reasons),
            "fallback_reason": "; ".join(fallback_reasons),
        }


# =========================
# Load unmatched DBLP papers
# =========================
unmatched_df = pd.read_csv(UNMATCHED_CSV)

unmatched_df["doi_norm"] = unmatched_df["doi"].map(normalize_doi)
unmatched_df["title_norm"] = unmatched_df["title"].map(normalize_title)
unmatched_df["year_norm"] = pd.to_numeric(unmatched_df["year"], errors="coerce")

doi_df = unmatched_df[unmatched_df["doi_norm"] != ""].copy()
title_df = unmatched_df[unmatched_df["doi_norm"] == ""].copy()

print(f"Unmatched papers: {len(unmatched_df):,}")
print(f"With DOI, use DOI lookup: {len(doi_df):,}")
print(f"Without DOI, use title lookup: {len(title_df):,}")


# =========================
# Load cache
# =========================
cache = load_cache(REFETCH_CACHE_JSON)
cache.setdefault("doi", {})
cache.setdefault("title", {})


# =========================
# Step 1. DOI batch lookup
# =========================
unique_dois = sorted(doi_df["doi_norm"].dropna().unique().tolist())

dois_to_fetch = [
    doi for doi in unique_dois
    if doi and should_fetch_cache_entry(cache["doi"].get(doi))
]

print(f"Unique DOI values: {len(unique_dois):,}")
print(f"DOI values to fetch: {len(dois_to_fetch):,}")

for batch_idx, doi_batch in enumerate(chunk_list(dois_to_fetch, DOI_BATCH_SIZE), start=1):
    print(f"[DOI batch {batch_idx}] Fetching {len(doi_batch)} DOIs")

    doi_values = [
        format_openalex_doi(doi)
        for doi in doi_batch
        if format_openalex_doi(doi)
    ]

    if not doi_values:
        continue

    filter_value = "doi:" + "|".join(doi_values)

    try:
        data = openalex_get({
            "filter": filter_value,
            "per_page": len(doi_values),
        })

        results = data.get("results", [])

        result_by_doi = {}
        for item in results:
            item_doi = normalize_doi(item.get("doi"))
            if item_doi:
                result_by_doi[item_doi] = item

        for doi in doi_batch:
            item = result_by_doi.get(normalize_doi(doi))

            if item is not None:
                cache["doi"][doi] = {
                    "status": "found",
                    "found": True,
                    "item": item,
                    "query_type": "doi",
                    "query": doi,
                    "method": "batch_doi",
                    "error": None,
                }
            else:
                cache["doi"][doi] = {
                    "status": "not_found",
                    "found": False,
                    "item": None,
                    "query_type": "doi",
                    "query": doi,
                    "method": "batch_doi",
                    "error": None,
                }

    except Exception as exc:
        print(f"[Warning] DOI batch failed: {exc}")
        print("Falling back to single DOI lookup...")

        for doi in doi_batch:
            try:
                item = fetch_openalex_by_single_doi(doi)

                if item is not None:
                    cache["doi"][doi] = {
                        "status": "found",
                        "found": True,
                        "item": item,
                        "query_type": "doi",
                        "query": doi,
                        "method": "single_doi_fallback",
                        "error": None,
                    }
                else:
                    cache["doi"][doi] = {
                        "status": "not_found",
                        "found": False,
                        "item": None,
                        "query_type": "doi",
                        "query": doi,
                        "method": "single_doi_fallback",
                        "error": None,
                    }

            except Exception as single_exc:
                print(f"[Warning] Single DOI failed: {doi} | {single_exc}")

                cache["doi"][doi] = {
                    "status": "failed",
                    "found": False,
                    "item": None,
                    "query_type": "doi",
                    "query": doi,
                    "method": "single_doi_fallback",
                    "error": str(single_exc),
                }

            time.sleep(0.1)

    save_cache(cache, REFETCH_CACHE_JSON)
    time.sleep(SLEEP_SECONDS + random.uniform(0, 0.2))


# =========================
# Step 2. Title lookup
# =========================
title_queries = []

for row_idx, row in title_df.iterrows():
    title = row.get("title", "")
    title_norm = row.get("title_norm", "")
    year = row.get("year_norm", None)
    year_value = safe_year_value(year)

    if not title_norm:
        continue
    
    
    
    cache_key = f"{conference_name}|{year_value if year_value is not None else ''}|{title_norm}"

    cached_title = cache["title"].get(cache_key)

    already_found = cached_title and (
        cached_title.get("status") == "found"
        or cached_title.get("found") is True
    )

    if already_found:
        continue

    strategy_is_current = (
        cached_title
        and cached_title.get("strategy_version") == TITLE_STRATEGY_VERSION
    )

    if strategy_is_current and not should_fetch_cache_entry(cached_title):
        continue


    title_queries.append({
        "row_idx": row_idx,
        "cache_key": cache_key,
        "title": title,
        "year": year_value,
    })

print(f"Title queries to fetch: {len(title_queries):,}")


def fetch_title_query(q):
    result = get_best_title_match(
        query_title=q["title"],
        year=q["year"],
    )

    return q["cache_key"], {
        "status": result["status"],
        "found": result["status"] == "found",
        "item": result["item"],
        "query_type": "title",
        "query": q["title"],
        "year": q["year"],
        "match_method": result["match_method"],
        "error": result["error"],
        "title_similarity": result["title_similarity"],
        "year_difference": result["year_difference"],
        "fallback_reason": result["fallback_reason"],
        "strategy_version": TITLE_STRATEGY_VERSION,
    }


with ThreadPoolExecutor(max_workers=TITLE_MAX_WORKERS) as executor:
    futures = [
        executor.submit(fetch_title_query, q)
        for q in title_queries
    ]

    for i, future in enumerate(as_completed(futures), start=1):
        try:
            cache_key, result = future.result()
            cache["title"][cache_key] = result

        except Exception as exc:
            print(f"[Warning] Title query failed outside request wrapper: {exc}")

        if i % 20 == 0:
            save_cache(cache, REFETCH_CACHE_JSON)
            print(f"Saved title cache after {i:,} title queries")

        time.sleep(SLEEP_SECONDS)

save_cache(cache, REFETCH_CACHE_JSON)


# =========================
# Step 3. Build outputs
# =========================
refetched_with_abstract = []
refetched_without_abstract = []
match_rows = []
still_unmatched_rows = []

seen_openalex_ids = set()


def add_matched_item(row_idx, row, item, match_method):
    openalex_id = item.get("id", "")
    abstract_available = has_abstract(item)
    query_year = safe_year_value(row.get("year"))
    openalex_year = safe_year_value(item.get("publication_year"))
    year_difference = (
        abs(openalex_year - query_year)
        if openalex_year is not None and query_year is not None
        else None
    )
    title_similarity = SequenceMatcher(
        None,
        normalize_title(row.get("title")),
        normalize_title(item.get("title")),
    ).ratio()

    match_rows.append({
        "dblp_row_index": row_idx,
        "conference": row.get("conference"),
        "year": row.get("year"),
        "title": row.get("title"),
        "doi": row.get("doi"),
        "openalex_id": openalex_id,
        "openalex_doi": item.get("doi"),
        "openalex_title": item.get("title"),
        "openalex_year": openalex_year,
        "year_difference": year_difference,
        "title_similarity": title_similarity,
        "match_method": match_method,
        "has_abstract": abstract_available,
    })

    if openalex_id and openalex_id in seen_openalex_ids:
        return

    if openalex_id:
        seen_openalex_ids.add(openalex_id)

    if abstract_available:
        refetched_with_abstract.append(item)
    else:
        refetched_without_abstract.append(item)


for row_idx, row in unmatched_df.iterrows():
    doi_norm = row.get("doi_norm", "")
    title_norm = row.get("title_norm", "")
    year_value = safe_year_value(row.get("year_norm", None))

    matched_item = None
    match_method = None

    if doi_norm:
        result = cache["doi"].get(doi_norm, {})

        if result.get("status") == "found" and result.get("item"):
            matched_item = result["item"]
            match_method = "openalex_api_doi"

    else:
        cache_key = f"{conference_name}|{year_value if year_value is not None else ''}|{title_norm}"
        result = cache["title"].get(cache_key, {})

        if result.get("status") == "found" and result.get("item"):
            matched_item = result["item"]
            match_method = f"openalex_api_{result.get('match_method', 'title')}"

    if matched_item is not None:
        add_matched_item(row_idx, row, matched_item, match_method)

    else:
        out = row.to_dict()
        if doi_norm:
            cached = cache["doi"].get(doi_norm, {})
        else:
            cache_key = f"{conference_name}|{year_value if year_value is not None else ''}|{title_norm}"
            cached = cache["title"].get(cache_key, {})

        out["openalex_refetch_status"] = cached.get("status", "not_requested")
        out["openalex_refetch_error"] = cached.get("error")
        out["openalex_refetch_match_method"] = cached.get("match_method") or cached.get("method")
        out["openalex_refetch_fallback_reason"] = cached.get("fallback_reason")
        still_unmatched_rows.append(out)


refetch_match_map_df = pd.DataFrame(match_rows)
still_unmatched_df = pd.DataFrame(still_unmatched_rows)

print(f"Refetched matched rows: {len(refetch_match_map_df):,}")
print(f"Refetched unique OpenAlex records with abstract: {len(refetched_with_abstract):,}")
print(f"Refetched unique OpenAlex records without abstract: {len(refetched_without_abstract):,}")
print(f"Still unmatched rows: {len(still_unmatched_df):,}")

if len(unmatched_df) > 0:
    print(f"Refetch match rate: {len(refetch_match_map_df) / len(unmatched_df) * 100:.2f}%")

if len(refetch_match_map_df) > 0:
    print(
        "Abstract coverage among refetched matched rows: "
        f"{refetch_match_map_df['has_abstract'].mean() * 100:.2f}%"
    )


# =========================
# Save outputs
# =========================
with REFETCHED_WITH_ABSTRACT_JSONL.open("w", encoding="utf-8") as f:
    for item in refetched_with_abstract:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with REFETCHED_WITHOUT_ABSTRACT_JSONL.open("w", encoding="utf-8") as f:
    for item in refetched_without_abstract:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

refetch_match_map_df.to_csv(REFETCH_MATCH_MAP_CSV, index=False, encoding="utf-8-sig")
still_unmatched_df.to_csv(STILL_UNMATCHED_CSV, index=False, encoding="utf-8-sig")

print(f"Saved refetched OpenAlex records with abstract to: {REFETCHED_WITH_ABSTRACT_JSONL}")
print(f"Saved refetched OpenAlex records without abstract to: {REFETCHED_WITHOUT_ABSTRACT_JSONL}")
print(f"Saved refetch match map to: {REFETCH_MATCH_MAP_CSV}")
print(f"Saved still-unmatched DBLP papers to: {STILL_UNMATCHED_CSV}")


# =========================
# Diagnostics
# =========================
if not refetch_match_map_df.empty:
    display(
        refetch_match_map_df
        .groupby(["year", "match_method", "has_abstract"])
        .size()
        .reset_index(name="n")
        .sort_values(["year", "match_method", "has_abstract"])
    )

if not still_unmatched_df.empty:
    display(
        still_unmatched_df
        .groupby(["year", "track_type", "openalex_refetch_status"])
        .size()
        .reset_index(name="n_still_unmatched")
        .sort_values(["year", "track_type", "openalex_refetch_status"])
    )


Unmatched papers: 560
With DOI, use DOI lookup: 0
Without DOI, use title lookup: 560
Unique DOI values: 0
DOI values to fetch: 0
Title queries to fetch: 0
Refetched matched rows: 558
Refetched unique OpenAlex records with abstract: 451
Refetched unique OpenAlex records without abstract: 107
Still unmatched rows: 2
Refetch match rate: 99.64%
Abstract coverage among refetched matched rows: 80.82%
Saved refetched OpenAlex records with abstract to: /home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Train_Data/iclr_openalex_with_abstract.jsonl
Saved refetched OpenAlex records without abstract to: /home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Train_Data/iclr_openalex_without_abstract.jsonl
Saved refetch match map to: /home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Train_Data/iclr_match_map.csv
Saved still-unmatched DBLP papers to: /home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publi

,year,match_method,has_abstract,n
0,2019,openalex_api_autocomplete_title_exact,False,107
1,2019,openalex_api_autocomplete_title_exact,True,447
2,2019,openalex_api_search_title_exact,True,1
3,2019,openalex_api_search_title_fuzzy,True,3


,year,track_type,openalex_refetch_status,n_still_unmatched
0,2019,main_track,not_found,1
1,2019,non-main_track,not_found,1


### 补充2025ACL的摘要
从../Data/20260708_openalex/acl_dblp_openalex_match_map.csv和../Data/20260708_openalex/acl_refetch_match_map.csv中提取acl没有摘要的论文，然后获取他们的doi的最后一个目录分隔符“/”后面的字符串，并合https://aclanthology.org/拼接，得到如https://aclanthology.org/2025.acl-long.1/这样的网址，通过解析这个网址获取摘要

已补充到combine.jsonl中，其他文件不动

### 1-3. 合并匹配到的openalex jsonl和未匹配到但从openalex获取到的jsonl；计算各个会议 年份 track_type的openalex且有摘要的覆盖率

In [6]:
# =========================
# Merge OpenAlex records with abstracts and calculate track-level coverage
# =========================
import json
import re
import pandas as pd
from pathlib import Path


# =========================
# Config
# =========================
BASE_DIR = Path("/home/user/GSK/lily/science_of_science/NewDataset")

DBLP_DIR = BASE_DIR / "AI_impact_on_cs_publications/Data/20260706_dblp"
MATCH_DIR = BASE_DIR / "AI_impact_on_cs_publications/Data/20260708_openalex"
COVERAGE_OUT_DIR = BASE_DIR /  "20260709_coverage_summary"
COVERAGE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to None to process every conference with available DBLP files.
# Example: CONFERENCE_NAMES = ["AAAI", "ACL", "CVPR"]
CONFERENCE_NAMES = ["IJCAI"]

COVERAGE_BY_YEAR_CSV = COVERAGE_OUT_DIR / "all_conferences_track_abstract_coverage_by_year.csv"
COVERAGE_BY_YEAR_WIDE_CSV = COVERAGE_OUT_DIR / "all_conferences_track_abstract_coverage_by_year_wide.csv"
COVERAGE_SUMMARY_CSV = COVERAGE_OUT_DIR / "all_conferences_track_abstract_coverage_summary.csv"


# =========================
# Helpers
# =========================
def normalize_conference(conf):
    if conf is None or pd.isna(conf):
        return ""

    conf = str(conf).strip().upper()

    aliases = {
        "THEWEB": "WWW",
        "THE WEB CONF": "WWW",
        "WEB CONF": "WWW",
    }

    return aliases.get(conf, conf)


def normalize_doi(doi):
    if doi is None or pd.isna(doi):
        return ""

    doi = str(doi).strip().lower()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "")
    doi = doi.replace("http://dx.doi.org/", "")
    doi = doi.replace("doi:", "")
    return doi.strip().rstrip(".")


def normalize_title(title):
    if title is None or pd.isna(title):
        return ""

    title = str(title).lower()
    title = re.sub(r"<[^>]+>", " ", title)
    title = re.sub(r"&amp;", " and ", title)
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def parse_bool(value):
    if isinstance(value, bool):
        return value

    if value is None or pd.isna(value):
        return False

    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def make_paper_key(conference, year, title, doi):
    conf_norm = normalize_conference(conference)

    try:
        year_norm = int(float(year))
    except Exception:
        year_norm = ""

    doi_norm = normalize_doi(doi)
    title_norm = normalize_title(title)

    if doi_norm:
        return f"{conf_norm}|{year_norm}|doi|{doi_norm}"

    return f"{conf_norm}|{year_norm}|title|{title_norm}"


def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return ""

    word_positions = []

    for word, positions in inverted_index.items():
        for pos in positions:
            word_positions.append((pos, word))

    word_positions.sort(key=lambda x: x[0])
    return " ".join(word for _, word in word_positions)


def item_has_abstract(item):
    if not item:
        return False

    if item.get("reconstructed_abstract"):
        return True

    if item.get("abstract"):
        return True

    if item.get("abstract_inverted_index"):
        abstract_text = reconstruct_abstract(item.get("abstract_inverted_index"))
        if abstract_text:
            item["reconstructed_abstract"] = abstract_text
            return True

    return False


def read_jsonl(path):
    if not path.exists() or path.stat().st_size == 0:
        return []

    items = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))

    return items


def write_jsonl(items, path):
    with path.open("w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def get_available_conferences():
    conferences = []

    for path in sorted(DBLP_DIR.glob("*_dblp_2020_2025_all_tracks.csv")):
        conf = path.name.split("_dblp_2020_2025_all_tracks.csv")[0].upper()

        has_initial = (MATCH_DIR / f"{conf.lower()}_dblp_openalex_match_map.csv").exists()
        has_refetch = (MATCH_DIR / f"{conf.lower()}_refetch_match_map.csv").exists()

        if has_initial or has_refetch:
            conferences.append(conf)

    return conferences


def merge_openalex_records_with_abstract(conference_name):
    conf_lower = conference_name.lower()

    input_jsonl_paths = [
        MATCH_DIR / f"{conf_lower}_matched_openalex_with_abstract.jsonl",
        MATCH_DIR / f"{conf_lower}_refetched_openalex_with_abstract.jsonl",
    ]

    combined_items = []
    seen_ids = set()

    for path in input_jsonl_paths:
        for item in read_jsonl(path):
            if not item_has_abstract(item):
                continue

            record_id = item.get("id") or item.get("doi") or item.get("title")

            if record_id in seen_ids:
                continue

            seen_ids.add(record_id)
            combined_items.append(item)

    output_path = MATCH_DIR / f"{conf_lower}_combined_openalex_with_abstract.jsonl"
    write_jsonl(combined_items, output_path)

    return output_path, len(combined_items)


def load_dblp_rows(conference_name):
    conf_lower = conference_name.lower()
    dblp_path = DBLP_DIR / f"{conf_lower}_dblp_2020_2025_all_tracks.csv"

    if not dblp_path.exists():
        raise FileNotFoundError(f"Cannot find DBLP file: {dblp_path}")

    df = pd.read_csv(dblp_path)

    df["conference"] = df["conference"].map(normalize_conference)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

    df["paper_key"] = df.apply(
        lambda row: make_paper_key(
            row.get("conference"),
            row.get("year"),
            row.get("title"),
            row.get("doi")
        ),
        axis=1,
    )

    return df


def load_matched_rows_with_abstract(conference_name):
    conf_lower = conference_name.lower()

    match_map_paths = [
        MATCH_DIR / f"{conf_lower}_dblp_openalex_match_map.csv",
        MATCH_DIR / f"{conf_lower}_refetch_match_map.csv",
    ]

    frames = []

    for path in match_map_paths:
        if not path.exists() or path.stat().st_size == 0:
            continue

        df = pd.read_csv(path)

        if "has_abstract" not in df.columns:
            continue

        df["has_abstract_bool"] = df["has_abstract"].map(parse_bool)
        df = df[df["has_abstract_bool"]].copy()

        if df.empty:
            continue

        df["conference"] = df["conference"].map(normalize_conference)
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

        df["paper_key"] = df.apply(
            lambda row: make_paper_key(
                row.get("conference"),
                row.get("year"),
                row.get("title"),
                row.get("doi")
            ),
            axis=1,
        )

        df["source_match_map"] = path.name
        frames.append(df)

    if not frames:
        return pd.DataFrame(columns=["paper_key"])

    matched_df = pd.concat(frames, ignore_index=True)
    matched_df = matched_df.drop_duplicates(subset=["paper_key"])

    return matched_df


    

def compute_track_coverage(conference_name):
    conference_name = normalize_conference(conference_name)

    combined_jsonl_path, n_combined_records = merge_openalex_records_with_abstract(
        conference_name
    )

    dblp_df = load_dblp_rows(conference_name)
    matched_df = load_matched_rows_with_abstract(conference_name)

    covered_keys = set(matched_df["paper_key"].dropna().tolist())
    dblp_df["has_openalex_abstract"] = dblp_df["paper_key"].isin(covered_keys)

    covered_rows_path = MATCH_DIR / f"{conference_name.lower()}_covered_dblp_rows_with_openalex_abstract.csv"

    dblp_df[dblp_df["has_openalex_abstract"]].to_csv(
        covered_rows_path,
        index=False,
        encoding="utf-8-sig",
    )

    coverage_df = (
        dblp_df
        .groupby(["conference", "year", "track_type"], dropna=False)
        .agg(
            n_dblp_papers=("paper_key", "count"),
            n_with_openalex_abstract=("has_openalex_abstract", "sum"),
        )
        .reset_index()
    )

    coverage_df["coverage_rate"] = (
        coverage_df["n_with_openalex_abstract"]
        / coverage_df["n_dblp_papers"]
        * 100
    ).round(2)

    coverage_df["combined_openalex_records_with_abstract"] = n_combined_records
    coverage_df["combined_openalex_jsonl"] = str(combined_jsonl_path)
    coverage_df["covered_dblp_rows_csv"] = str(covered_rows_path)

    return coverage_df


# =========================
# Run coverage calculation
# =========================
if CONFERENCE_NAMES is None:
    conference_names = get_available_conferences()
else:
    conference_names = [normalize_conference(conf) for conf in CONFERENCE_NAMES]

print(f"Conferences to process: {conference_names}")

coverage_frames = []

for conf in conference_names:
    print(f"Processing {conf}...")
    coverage_frames.append(compute_track_coverage(conf))

coverage_all = pd.concat(coverage_frames, ignore_index=True)

coverage_all = (
    coverage_all
    .sort_values(["conference", "year", "track_type"])
    .reset_index(drop=True)
)

coverage_all.to_csv(
    COVERAGE_BY_YEAR_CSV,
    index=False,
    encoding="utf-8-sig"
)

coverage_wide = (
    coverage_all
    .pivot_table(
        index=["conference", "year"],
        columns="track_type",
        values=[
            "n_dblp_papers",
            "n_with_openalex_abstract",
            "coverage_rate",
        ],
        fill_value=0,
        aggfunc="first",
    )
)

coverage_wide.columns = [
    f"{metric}_{track}"
    for metric, track in coverage_wide.columns
]

coverage_wide = coverage_wide.reset_index()

coverage_wide.to_csv(
    COVERAGE_BY_YEAR_WIDE_CSV,
    index=False,
    encoding="utf-8-sig"
)

coverage_summary = (
    coverage_all
    .groupby(["conference", "track_type"], dropna=False)
    .agg(
        n_dblp_papers=("n_dblp_papers", "sum"),
        n_with_openalex_abstract=("n_with_openalex_abstract", "sum"),
    )
    .reset_index()
)

coverage_summary["coverage_rate"] = (
    coverage_summary["n_with_openalex_abstract"]
    / coverage_summary["n_dblp_papers"]
    * 100
).round(2)

coverage_summary.to_csv(
    COVERAGE_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved coverage by year to: {COVERAGE_BY_YEAR_CSV}")
print(f"Saved wide coverage table to: {COVERAGE_BY_YEAR_WIDE_CSV}")
print(f"Saved coverage summary to: {COVERAGE_SUMMARY_CSV}")

display(coverage_all)
display(coverage_wide)
display(coverage_summary)

Conferences to process: ['IJCAI']
Processing IJCAI...


Saved coverage by year to: /home/user/GSK/lily/science_of_science/NewDataset/20260709_coverage_summary/all_conferences_track_abstract_coverage_by_year.csv
Saved wide coverage table to: /home/user/GSK/lily/science_of_science/NewDataset/20260709_coverage_summary/all_conferences_track_abstract_coverage_by_year_wide.csv
Saved coverage summary to: /home/user/GSK/lily/science_of_science/NewDataset/20260709_coverage_summary/all_conferences_track_abstract_coverage_summary.csv


,conference,year,track_type,n_dblp_papers,n_with_openalex_abstract,coverage_rate,combined_openalex_records_with_abstract,combined_openalex_jsonl,covered_dblp_rows_csv
0,IJCAI,2020,main_track,592,591,99.83,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
1,IJCAI,2020,non-main_track,228,189,82.89,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
2,IJCAI,2021,main_track,586,586,100.00,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
3,IJCAI,2021,non-main_track,174,144,82.76,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
4,IJCAI,2022,main_track,679,679,100.00,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
5,IJCAI,2022,non-main_track,250,189,75.60,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
6,IJCAI,2023,main_track,639,639,100.00,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
7,IJCAI,2023,non-main_track,214,208,97.20,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
8,IJCAI,2024,main_track,790,790,100.00,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...
9,IJCAI,2024,non-main_track,266,258,96.99,5553,/home/user/GSK/lily/science_of_science/NewData...,/home/user/GSK/lily/science_of_science/NewData...


,conference,year,coverage_rate_main_track,coverage_rate_non-main_track,n_dblp_papers_main_track,n_dblp_papers_non-main_track,n_with_openalex_abstract_main_track,n_with_openalex_abstract_non-main_track
0,IJCAI,2020,99.83,82.89,592,228,591,189
1,IJCAI,2021,100.00,82.76,586,174,586,144
2,IJCAI,2022,100.00,75.60,679,250,679,189
3,IJCAI,2023,100.00,97.20,639,214,639,208
4,IJCAI,2024,100.00,96.99,790,266,790,258
5,IJCAI,2025,100.00,100.00,1014,266,1014,266


,conference,track_type,n_dblp_papers,n_with_openalex_abstract,coverage_rate
0,IJCAI,main_track,4300,4299,99.98
1,IJCAI,non-main_track,1398,1254,89.70


In [8]:
import json
import re
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd


BASE_DIR = Path("/home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Data")
DBLP_DIR = BASE_DIR / "20260706_dblp"
COMBINE_DIR = BASE_DIR / "20260708_openalex"
OUT_DIR = BASE_DIR / "20260711_coverage_summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFERENCES = ["AAAI", "ACL", "CVPR", "EMNLP", "ICLR", "ICML", "IJCAI", "KDD", "SIGIR", "WWW"]


def normalize_conference(x):
    x = str(x or "").strip().upper()
    aliases = {
        "THEWEB": "WWW",
        "THE WEB CONF": "WWW",
        "WEB CONF": "WWW",
    }
    return aliases.get(x, x)


def normalize_doi(x):
    x = str(x or "").strip().lower()
    x = x.replace("https://doi.org/", "")
    x = x.replace("http://doi.org/", "")
    x = x.replace("https://dx.doi.org/", "")
    x = x.replace("http://dx.doi.org/", "")
    x = x.replace("doi:", "")
    x = x.strip().rstrip(".")
    if x in {"", "nan", "none"}:
        return ""
    return x


def normalize_title(x):
    x = str(x or "").lower()
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"&amp;", " and ", x)
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    if x in {"", "nan", "none"}:
        return ""
    return x


def safe_year(x):
    try:
        if x is None or str(x).lower() in {"nan", "none", ""}:
            return None
        return int(float(x))
    except Exception:
        return None


def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return ""

    pairs = []
    for word, positions in inverted_index.items():
        for pos in positions:
            pairs.append((pos, word))

    pairs.sort(key=lambda x: x[0])
    return " ".join(word for _, word in pairs)


def has_abstract(item):
    if str(item.get("reconstructed_abstract") or "").strip():
        return True

    if str(item.get("abstract") or "").strip():
        return True

    if item.get("abstract_inverted_index"):
        return bool(reconstruct_abstract(item["abstract_inverted_index"]).strip())

    return False


def get_candidate_metadata(item, fallback_conf):
    candidates = []

    source = item.get("dblp_source")
    if isinstance(source, dict):
        candidates.append({
            "conference": source.get("conference") or fallback_conf,
            "year": source.get("year"),
            "title": source.get("title"),
            "doi": source.get("doi"),
            "source": "dblp_source",
        })

    candidates.append({
        "conference": item.get("conference") or item.get("Conference") or fallback_conf,
        "year": item.get("publication_year") or item.get("year") or item.get("Year"),
        "title": item.get("title") or item.get("display_name") or item.get("openalex_title"),
        "doi": item.get("doi") or item.get("openalex_doi"),
        "source": "item",
    })

    return candidates


def build_combine_indexes():
    doi_index = defaultdict(set)
    title_year_index = defaultdict(set)
    title_only_tmp = defaultdict(set)

    combined_stats = []

    record_id = 0

    for conf in CONFERENCES:
        path = COMBINE_DIR / f"{conf.lower()}_combined_openalex_with_abstract.jsonl"

        n_records = 0
        n_with_abstract = 0

        if not path.exists():
            print(f"[Missing combine] {path}")
            continue

        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                n_records += 1
                item = json.loads(line)

                if not has_abstract(item):
                    continue

                n_with_abstract += 1

                for meta in get_candidate_metadata(item, fallback_conf=conf):
                    c = normalize_conference(meta["conference"])
                    y = safe_year(meta["year"])
                    t = normalize_title(meta["title"])
                    d = normalize_doi(meta["doi"])

                    if d:
                        doi_index[d].add(record_id)

                    if c and y is not None and t:
                        title_year_index[(c, y, t)].add(record_id)

                    if c and t:
                        title_only_tmp[(c, t)].add(record_id)

                record_id += 1

        combined_stats.append({
            "conference": conf,
            "n_combined_records": n_records,
            "n_combined_records_with_abstract": n_with_abstract,
            "combined_jsonl": str(path),
        })

    title_only_index = {
        key: ids
        for key, ids in title_only_tmp.items()
        if len(ids) == 1
    }

    return doi_index, title_year_index, title_only_index, pd.DataFrame(combined_stats)


doi_index, title_year_index, title_only_index, combined_stats = build_combine_indexes()


all_dblp_rows = []

for conf in CONFERENCES:
    path = DBLP_DIR / f"{conf.lower()}_dblp_2020_2025_all_tracks.csv"

    if not path.exists():
        print(f"[Missing DBLP] {path}")
        continue

    df = pd.read_csv(path)
    df["conference"] = df["conference"].map(normalize_conference)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["doi_norm"] = df["doi"].map(normalize_doi)
    df["title_norm"] = df["title"].map(normalize_title)

    all_dblp_rows.append(df)

dblp_df = pd.concat(all_dblp_rows, ignore_index=True)


def match_dblp_row(row):
    conf = normalize_conference(row["conference"])
    year = safe_year(row["year"])
    doi = row["doi_norm"]
    title = row["title_norm"]

    if doi and doi in doi_index:
        return True, "doi"

    if conf and year is not None and title and (conf, year, title) in title_year_index:
        return True, "conference_year_title"

    if conf and title and (conf, title) in title_only_index:
        return True, "conference_title_unique"

    return False, "unmatched"


match_results = dblp_df.apply(match_dblp_row, axis=1, result_type="expand")
dblp_df["has_openalex_abstract"] = match_results[0]
dblp_df["coverage_match_method"] = match_results[1]


coverage_by_year_track = (
    dblp_df
    .groupby(["conference", "year", "track_type"], dropna=False)
    .agg(
        n_dblp_papers=("title", "count"),
        n_with_openalex_abstract=("has_openalex_abstract", "sum"),
    )
    .reset_index()
)

coverage_by_year_track["coverage_rate"] = (
    coverage_by_year_track["n_with_openalex_abstract"]
    / coverage_by_year_track["n_dblp_papers"]
    * 100
).round(2)


coverage_by_year = (
    dblp_df
    .groupby(["conference", "year"], dropna=False)
    .agg(
        n_dblp_papers=("title", "count"),
        n_with_openalex_abstract=("has_openalex_abstract", "sum"),
    )
    .reset_index()
)

coverage_by_year["coverage_rate"] = (
    coverage_by_year["n_with_openalex_abstract"]
    / coverage_by_year["n_dblp_papers"]
    * 100
).round(2)


coverage_summary = (
    dblp_df
    .groupby(["conference", "track_type"], dropna=False)
    .agg(
        n_dblp_papers=("title", "count"),
        n_with_openalex_abstract=("has_openalex_abstract", "sum"),
    )
    .reset_index()
)

coverage_summary["coverage_rate"] = (
    coverage_summary["n_with_openalex_abstract"]
    / coverage_summary["n_dblp_papers"]
    * 100
).round(2)


match_method_counts = (
    dblp_df
    .groupby(["conference", "coverage_match_method"])
    .size()
    .reset_index(name="n")
)


matched_rows = dblp_df[dblp_df["has_openalex_abstract"]].copy()
unmatched_rows = dblp_df[~dblp_df["has_openalex_abstract"]].copy()


coverage_by_year_track.to_csv(
    OUT_DIR / "dblp_based_coverage_by_conference_year_track.csv",
    index=False,
    encoding="utf-8-sig",
)

coverage_by_year.to_csv(
    OUT_DIR / "dblp_based_coverage_by_conference_year.csv",
    index=False,
    encoding="utf-8-sig",
)

coverage_summary.to_csv(
    OUT_DIR / "dblp_based_coverage_summary_by_conference_track.csv",
    index=False,
    encoding="utf-8-sig",
)

match_method_counts.to_csv(
    OUT_DIR / "dblp_based_coverage_match_method_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

combined_stats.to_csv(
    OUT_DIR / "combined_jsonl_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

matched_rows.to_csv(
    OUT_DIR / "dblp_rows_matched_to_combined_with_abstract.csv",
    index=False,
    encoding="utf-8-sig",
)

unmatched_rows.to_csv(
    OUT_DIR / "dblp_rows_unmatched_to_combined_with_abstract.csv",
    index=False,
    encoding="utf-8-sig",
)


overall_n = len(dblp_df)
overall_matched = int(dblp_df["has_openalex_abstract"].sum())
overall_rate = round(overall_matched / overall_n * 100, 2)

print(f"Overall DBLP-based coverage: {overall_matched}/{overall_n} = {overall_rate}%")

print("\nCoverage by conference-year-track:")
display(coverage_by_year_track)

print("\nCoverage by conference-year:")
display(coverage_by_year)

print("\nCoverage summary by conference-track:")
display(coverage_summary)

print("\nMatch method counts:")
display(match_method_counts)

print("\nCombined jsonl counts:")
display(combined_stats)

Overall DBLP-based coverage: 90691/91890 = 98.7%

Coverage by conference-year-track:


,conference,year,track_type,n_dblp_papers,n_with_openalex_abstract,coverage_rate
0,AAAI,2020,main_track,1584,1583,99.94
1,AAAI,2020,non-main_track,321,279,86.92
2,AAAI,2021,main_track,1654,1654,100.00
3,AAAI,2021,non-main_track,409,309,75.55
4,AAAI,2022,main_track,1319,1319,100.00
...,...,...,...,...,...,...
108,WWW,2023,non-main_track,297,297,100.00
109,WWW,2024,main_track,405,384,94.81
110,WWW,2024,non-main_track,373,371,99.46
111,WWW,2025,main_track,409,400,97.80



Coverage by conference-year:


,conference,year,n_dblp_papers,n_with_openalex_abstract,coverage_rate
0,AAAI,2020,1905,1862,97.74
1,AAAI,2021,2063,1963,95.15
2,AAAI,2022,1680,1628,96.90
3,AAAI,2023,2077,2031,97.79
4,AAAI,2024,2882,2866,99.44
5,AAAI,2025,3509,3490,99.46
6,ACL,2020,887,886,99.89
7,ACL,2021,1246,1240,99.52
8,ACL,2022,1105,1103,99.82
9,ACL,2023,2148,2147,99.95



Coverage summary by conference-track:


,conference,track_type,n_dblp_papers,n_with_openalex_abstract,coverage_rate
0,AAAI,main_track,11663,11662,99.99
1,AAAI,non-main_track,2453,2178,88.79
2,ACL,main_track,5893,5893,100.00
3,ACL,non-main_track,4847,4821,99.46
4,CVPR,main_track,13136,13125,99.92
5,CVPR,non-main_track,3786,3716,98.15
6,EMNLP,main_track,6551,6542,99.86
7,EMNLP,non-main_track,5752,5742,99.83
8,ICLR,main_track,10178,10092,99.16
9,ICLR,non-main_track,413,91,22.03



Match method counts:


,conference,coverage_match_method,n
0,AAAI,conference_year_title,30
1,AAAI,doi,13810
2,AAAI,unmatched,276
3,ACL,conference_title_unique,104
4,ACL,conference_year_title,236
5,ACL,doi,10374
6,ACL,unmatched,26
7,CVPR,conference_year_title,650
8,CVPR,doi,16191
9,CVPR,unmatched,81



Combined jsonl counts:


,conference,n_combined_records,n_combined_records_with_abstract,combined_jsonl
0,AAAI,13840,13840,/home/user/GSK/lily/science_of_science/NewData...
1,ACL,10714,10714,/home/user/GSK/lily/science_of_science/NewData...
2,CVPR,16834,16834,/home/user/GSK/lily/science_of_science/NewData...
3,EMNLP,12284,12284,/home/user/GSK/lily/science_of_science/NewData...
4,ICLR,10215,10215,/home/user/GSK/lily/science_of_science/NewData...
5,ICML,11318,11318,/home/user/GSK/lily/science_of_science/NewData...
6,IJCAI,5553,5553,/home/user/GSK/lily/science_of_science/NewData...
7,KDD,3409,3409,/home/user/GSK/lily/science_of_science/NewData...
8,SIGIR,2575,2575,/home/user/GSK/lily/science_of_science/NewData...
9,WWW,3998,3998,/home/user/GSK/lily/science_of_science/NewData...
